#1. Convert protobuf file to readable csv file in volume

In [0]:
import gtfs_realtime_pb2

def gtfs_to_rows(feed):
    rows = []

    for entity in feed.entity:
        row = {
            "entity_id": entity.id,
        }

        if entity.HasField("vehicle"):
            v = entity.vehicle
            row.update({
                "type": "vehicle",
                "vehicle_id": v.vehicle.id if v.vehicle.id else "",
                "trip_id": v.trip.trip_id,
                "route_id": v.trip.route_id,
                "latitude": v.position.latitude,
                "longitude": v.position.longitude,
                "speed": v.position.speed,
                "bearing": v.position.bearing,
                "timestamp": v.timestamp,
            })

        elif entity.HasField("trip_update"):
            t = entity.trip_update
            row.update({
                "type": "trip_update",
                "trip_id": t.trip.trip_id,
                "route_id": t.trip.route_id,
                "vehicle_id": t.vehicle.id if t.vehicle.id else "",
                "timestamp": t.timestamp,
            })

        elif entity.HasField("alert"):
            row.update({
                "type": "alert",
                "header": str(entity.alert.header_text),
                "description": str(entity.alert.description_text),
            })

        rows.append(row)

    return rows

file_path = ''
for file_object in dbutils.fs.ls('/Volumes/warsaw_gtfs/default/text_data'):
    if file_object.name.endswith('.pb'):
        file_path = '/Volumes/warsaw_gtfs/default/text_data/' + file_object.name
        break

feed = gtfs_realtime_pb2.FeedMessage()

with open(file_path, 'rb') as f:
    feed.ParseFromString(f.read())

rows = gtfs_to_rows(feed)

realtime_df = spark.createDataFrame(rows)
realtime_df.display()

In [0]:
import pyspark.sql.functions as F


realtime_df.filter(F.col('speed') > 0).display()